

En esta actividad se implementa una solución completa para clasificar imágenes del dataset MNIST usando únicamente PyTorch para el modelo, entrenamiento y evaluación. También se decidió usar la CPU por que no termine de entender como usar cuda pero por si acaso hay una comprobacion en un if, ademas marco la semilla en una base solo para guardar y tener un resultado "fijo" temporlmente.

In [32]:
import gzip
from pathlib import Path
import numpy as np
import torch



torch.manual_seed(42)
np.random.seed(42)


dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo en uso:", dispositivo)

Dispositivo en uso: cpu


aqui cargo el set de imagenes, por que lo hice con la ruta absoluta que incluye mi usuario y eso es que perdi la costumbre de programar localmente y preferi hacerlo asi por si acaso, los nombres de las variables y funciones son asi en español para no confindirme en lo que hacen, se que hay que usar torch para todo pero no creo que se refiera a todo todo solo a los calculos y los esenciales, asi que por eso pues en estas partes elementos de otras librerias debido a ello mas adelante se vera reflejado el uso exclusivo de torch, use como base para esto el codigo de usted en el ejemplo

In [33]:
def obteneretiquetas(ruta):
    with gzip.open(ruta, "rb") as datos:
        etiquetas = datos.read()[8:]
        return np.frombuffer(etiquetas, dtype=np.uint8)


def obtenerimg(ruta):
    with gzip.open(ruta, "rb") as datos:
        _ = int.from_bytes(datos.read(4), "big")
        numeroimg = int.from_bytes(datos.read(4), "big")
        filas = int.from_bytes(datos.read(4), "big")
        columnas = int.from_bytes(datos.read(4), "big")
        img = datos.read()
        return np.frombuffer(img, dtype=np.uint8).reshape((numeroimg, filas, columnas))


def cargardatasetmnist(rutamnist):
    imgentrenamientovalidacion = obtenerimg(Path(rutamnist) / "train-images-idx3-ubyte.gz")
    etiquetasentrenamientovalidacion = obteneretiquetas(Path(rutamnist) / "train-labels-idx1-ubyte.gz")

    imgentrenamiento = imgentrenamientovalidacion[:50000]
    etiquetasentrenamiento = etiquetasentrenamientovalidacion[:50000]

    imgvalidacion = imgentrenamientovalidacion[50000:]
    etiquetasvalidacion = etiquetasentrenamientovalidacion[50000:]

    imgprueba = obtenerimg(Path(rutamnist) / "t10k-images-idx3-ubyte.gz")
    etiquetasprueba = obteneretiquetas(Path(rutamnist) / "t10k-labels-idx1-ubyte.gz")

    return (
        imgentrenamiento,
        etiquetasentrenamiento,
        imgvalidacion,
        etiquetasvalidacion,
        imgprueba,
        etiquetasprueba
    )


rutamnist = r"C:\Users\Rafael\Documents\GitHub\computo-inteligente-Rafael-Negrete-Leyva\mnist"

imgentrenamientonp, etiquetasentrenamientonp, imgvalidacionnp, etiquetasvalidacionnp, imgpruebanp, etiquetaspruebanp = cargardatasetmnist(rutamnist)

print("Train:", imgentrenamientonp.shape, etiquetasentrenamientonp.shape)
print("Validación:", imgvalidacionnp.shape, etiquetasvalidacionnp.shape)
print("Prueba:", imgpruebanp.shape, etiquetaspruebanp.shape)

Train: (50000, 28, 28) (50000,)
Validación: (10000, 28, 28) (10000,)
Prueba: (10000, 28, 28) (10000,)


Se normalizan los pixeles al rango de 0 a 1 para facilitar el entrenamiento. También se convierten las etiquetas a tipo long, porque esa es la forma que necesita la función de pérdida para clasificación multiclase.

In [34]:
imgentrenamiento = torch.from_numpy(imgentrenamientonp).float() / 255.0
imgvalidacion = torch.from_numpy(imgvalidacionnp).float() / 255.0
imgprueba = torch.from_numpy(imgpruebanp).float() / 255.0

etiquetasentrenamiento = torch.from_numpy(etiquetasentrenamientonp).long()
etiquetasvalidacion = torch.from_numpy(etiquetasvalidacionnp).long()
etiquetasprueba = torch.from_numpy(etiquetaspruebanp).long()

datasetentrenamiento = torch.utils.data.TensorDataset(imgentrenamiento, etiquetasentrenamiento)
datasetvalidacion = torch.utils.data.TensorDataset(imgvalidacion, etiquetasvalidacion)
datasetprueba = torch.utils.data.TensorDataset(imgprueba, etiquetasprueba)

print("Elementos entrenamiento:", len(datasetentrenamiento))
print("Elementos validación:", len(datasetvalidacion))
print("Elementos prueba:", len(datasetprueba))

Elementos entrenamiento: 50000
Elementos validación: 10000
Elementos prueba: 10000


La red se construye usando únicamente capas Linear y ReLU

In [35]:
class Redneuronalmulticlase(torch.nn.Module):
    def __init__(self, capasocultas):
        super().__init__()

        capas = []
        tamanoentrada = 28 * 28

        for tamanosalida in capasocultas:
            capas.append(torch.nn.Linear(tamanoentrada, tamanosalida))
            capas.append(torch.nn.ReLU())
            tamanoentrada = tamanosalida

        capas.append(torch.nn.Linear(tamanoentrada, 10))
        self.red = torch.nn.Sequential(*capas)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.red(x)

separaron varias funciones auxiliares hace más fácil entrenar y usar varios modelos

In [36]:
def crearcargadores(batchsizeentrenamiento, batchsizeevaluacion=1024):
    cargadorentrenamiento = torch.utils.data.DataLoader(
        datasetentrenamiento,
        batch_size=batchsizeentrenamiento,
        shuffle=True
    )

    cargadorvalidacion = torch.utils.data.DataLoader(
        datasetvalidacion,
        batch_size=batchsizeevaluacion,
        shuffle=False
    )

    cargadorprueba = torch.utils.data.DataLoader(
        datasetprueba,
        batch_size=batchsizeevaluacion,
        shuffle=False
    )

    return cargadorentrenamiento, cargadorvalidacion, cargadorprueba


def crearoptimizador(nombreoptimizador, parametrosmodelo, learningrate):
    nombre = nombreoptimizador.lower()

    if nombre == "adam":
        return torch.optim.Adam(parametrosmodelo, lr=learningrate)

    if nombre == "sgd":
        return torch.optim.SGD(parametrosmodelo, lr=learningrate, momentum=0.9)

    raise ValueError(f"Optimizador no soportado: {nombreoptimizador}")


def evaluarmodelo(modelo, cargador, funcionperdida, devolverpredicciones=False):
    modelo.eval()

    perdidaacumulada = 0.0
    totalejemplos = 0
    totalcorrectos = 0

    prediccionestotales = []
    etiquetastotales = []

    with torch.no_grad():
        for imglote, etiquetaslote in cargador:
            imglote = imglote.to(dispositivo)
            etiquetaslote = etiquetaslote.to(dispositivo)

            logits = modelo(imglote)
            loss = funcionperdida(logits, etiquetaslote)

            predicciones = torch.argmax(logits, dim=1)

            tamanolote = etiquetaslote.size(0)
            perdidaacumulada += loss.item() * tamanolote
            totalejemplos += tamanolote
            totalcorrectos += (predicciones == etiquetaslote).sum().item()

            if devolverpredicciones:
                prediccionestotales.append(predicciones.cpu())
                etiquetastotales.append(etiquetaslote.cpu())

    losspromedio = perdidaacumulada / totalejemplos
    accuracy = totalcorrectos / totalejemplos

    if devolverpredicciones:
        prediccionestotales = torch.cat(prediccionestotales)
        etiquetastotales = torch.cat(etiquetastotales)
        return losspromedio, accuracy, prediccionestotales, etiquetastotales

    return losspromedio, accuracy

implementa early stopping sin usar librerías externas curiosamente vimos en clase este tema el dia de hoy esto sirve para detener el entrenamiento cuando la mejora en la pérdida de validación es demasiado pequeña

In [37]:
def detenertemprano(historiallossvalidacion, patience, delta):
    if len(historiallossvalidacion) < patience + 1:
        return False

    losspasado = historiallossvalidacion[-(patience + 1)]
    lossactual = historiallossvalidacion[-1]
    mejora = losspasado - lossactual

    if 0 <= mejora < delta:
        return True

    return False

el entrenamiento por mini batches pesos y se guarda como esta siguiendo el procedimiento

In [38]:
def entrenarmodelo(configuracion):
    print("\n" + "=" * 80)
    print("Entrenando:", configuracion["nombre"])
    print("=" * 80)

    modelo = Redneuronalmulticlase(configuracion["capasocultas"]).to(dispositivo)
    funcionperdida = torch.nn.CrossEntropyLoss()
    optimizador = crearoptimizador(
        configuracion["optimizador"],
        modelo.parameters(),
        configuracion["learningrate"]
    )

    cargadorentrenamiento, cargadorvalidacion, _ = crearcargadores(
        batchsizeentrenamiento=configuracion["batchsize"]
    )

    historial = {
        "lossentrenamiento": [],
        "accuracyentrenamiento": [],
        "lossvalidacion": [],
        "accuracyvalidacion": []
    }

    mejoraccuracyvalidacion = -1.0
    mejorlossvalidacion = float("inf")
    mejorepoca = 0
    mejorestado = None

    for epoca in range(1, configuracion["epocas"] + 1):
        modelo.train()

        perdidaacumulada = 0.0
        totalejemplos = 0
        totalcorrectos = 0

        for imglote, etiquetaslote in cargadorentrenamiento:
            imglote = imglote.to(dispositivo)
            etiquetaslote = etiquetaslote.to(dispositivo)

            optimizador.zero_grad()
            logits = modelo(imglote)
            loss = funcionperdida(logits, etiquetaslote)
            loss.backward()
            optimizador.step()

            predicciones = torch.argmax(logits, dim=1)

            tamanolote = etiquetaslote.size(0)
            perdidaacumulada += loss.item() * tamanolote
            totalejemplos += tamanolote
            totalcorrectos += (predicciones == etiquetaslote).sum().item()

        lossentrenamiento = perdidaacumulada / totalejemplos
        accuracyentrenamiento = totalcorrectos / totalejemplos

        lossvalidacion, accuracyvalidacion = evaluarmodelo(
            modelo,
            cargadorvalidacion,
            funcionperdida
        )

        historial["lossentrenamiento"].append(lossentrenamiento)
        historial["accuracyentrenamiento"].append(accuracyentrenamiento)
        historial["lossvalidacion"].append(lossvalidacion)
        historial["accuracyvalidacion"].append(accuracyvalidacion)

        print(
            f"Época {epoca:02d} | "
            f"Loss train: {lossentrenamiento:.4f} | "
            f"Accuracy train: {accuracyentrenamiento:.4f} | "
            f"Loss val: {lossvalidacion:.4f} | "
            f"Accuracy val: {accuracyvalidacion:.4f}"
        )

        esmejormodelo = False

        if accuracyvalidacion > mejoraccuracyvalidacion:
            esmejormodelo = True
        elif accuracyvalidacion == mejoraccuracyvalidacion and lossvalidacion < mejorlossvalidacion:
            esmejormodelo = True

        if esmejormodelo:
            mejoraccuracyvalidacion = accuracyvalidacion
            mejorlossvalidacion = lossvalidacion
            mejorepoca = epoca
            mejorestado = {
                nombre: tensor.detach().cpu().clone()
                for nombre, tensor in modelo.state_dict().items()
            }

        if detenertemprano(
            historial["lossvalidacion"],
            configuracion["patience"],
            configuracion["delta"]
        ):
            print(
                f"Early stopping activado en la época {epoca} "
                f"con patience={configuracion['patience']} y delta={configuracion['delta']}"
            )
            break

    modelo.load_state_dict(mejorestado)

    return {
        "nombre": configuracion["nombre"],
        "modelo": modelo,
        "historial": historial,
        "mejoraccuracyvalidacion": mejoraccuracyvalidacion,
        "mejorlossvalidacion": mejorlossvalidacion,
        "mejorepoca": mejorepoca,
        "configuracion": configuracion
    }

La matriz de confusión y el resto de métricas se calculan manualmente usando PyTorch y Python base. con base no me refiero a python "vacio" si no a python sin mandar ninguna libreria externa como np que si bien parece la base de python pues casi siempre lo es ocupa ser llamado

In [39]:
def construirmatrizconfusion(etiquetasreales, etiquetaspredichas, numeroclases=10):
    matrizconfusion = torch.zeros((numeroclases, numeroclases), dtype=torch.int64)

    for etiquetareal, etiquetapredicha in zip(etiquetasreales, etiquetaspredichas):
        matrizconfusion[etiquetareal.long(), etiquetapredicha.long()] += 1

    return matrizconfusion


def calcularreportedesdematriz(matrizconfusion):
    matriz = matrizconfusion.float()

    verdaderospositivos = torch.diag(matriz)
    falsospositivos = matriz.sum(dim=0) - verdaderospositivos
    falsosnegativos = matriz.sum(dim=1) - verdaderospositivos

    precision = torch.zeros(matriz.size(0))
    recall = torch.zeros(matriz.size(0))
    f1score = torch.zeros(matriz.size(0))

    for clase in range(matriz.size(0)):
        denominadorprecision = verdaderospositivos[clase] + falsospositivos[clase]
        denominadorrecall = verdaderospositivos[clase] + falsosnegativos[clase]

        if denominadorprecision > 0:
            precision[clase] = verdaderospositivos[clase] / denominadorprecision

        if denominadorrecall > 0:
            recall[clase] = verdaderospositivos[clase] / denominadorrecall

        if precision[clase] + recall[clase] > 0:
            f1score[clase] = 2 * precision[clase] * recall[clase] / (precision[clase] + recall[clase])

    accuracyglobal = verdaderospositivos.sum() / matriz.sum()

    return {
        "accuracyglobal": accuracyglobal.item(),
        "precisionporclase": precision,
        "recallporclase": recall,
        "f1scoreporclase": f1score
    }


def imprimirreporteclasificacion(matrizconfusion):
    reporte = calcularreportedesdematriz(matrizconfusion)

    print("\nMatriz de confusión:")
    print(matrizconfusion)

    print("\nAccuracy global:")
    print(f"{reporte['accuracyglobal']:.6f}")

    print("\nReporte por clase:")
    print(f"{'Clase':<10}{'Precision':<15}{'Recall':<15}{'F1Score':<15}")

    for clase in range(matrizconfusion.size(0)):
        precision = reporte["precisionporclase"][clase].item()
        recall = reporte["recallporclase"][clase].item()
        f1score = reporte["f1scoreporclase"][clase].item()

        print(f"{clase:<10}{precision:<15.6f}{recall:<15.6f}{f1score:<15.6f}")

In [40]:
configuraciones = [
    {
        "nombre": "Modelo 1 con una capa oculta y Adam",
        "capasocultas": [128],
        "optimizador": "Adam",
        "batchsize": 128,
        "epocas": 200,
        "learningrate": 0.001,
        "patience": 3,
        "delta": 0.001
    },
    {
        "nombre": "Modelo 2 con dos capas ocultas y Adam",
        "capasocultas": [256, 128],
        "optimizador": "Adam",
        "batchsize": 256,
        "epocas": 250,
        "learningrate": 0.001,
        "patience": 3,
        "delta": 0.001
    },
    {
        "nombre": "Modelo 3 con tres capas ocultas y SGD",
        "capasocultas": [512, 256, 128],
        "optimizador": "SGD",
        "batchsize": 128,
        "epocas": 300,
        "learningrate": 0.05,
        "patience": 4,
        "delta": 0.001
    }
]

In [41]:
resultados = []

for configuracion in configuraciones:
    resultado = entrenarmodelo(configuracion)
    resultados.append(resultado)


Entrenando: Modelo 1 con una capa oculta y Adam
Época 01 | Loss train: 0.4455 | Accuracy train: 0.8825 | Loss val: 0.2318 | Accuracy val: 0.9367
Época 02 | Loss train: 0.2098 | Accuracy train: 0.9405 | Loss val: 0.1701 | Accuracy val: 0.9550
Época 03 | Loss train: 0.1553 | Accuracy train: 0.9550 | Loss val: 0.1441 | Accuracy val: 0.9588
Época 04 | Loss train: 0.1238 | Accuracy train: 0.9643 | Loss val: 0.1196 | Accuracy val: 0.9683
Época 05 | Loss train: 0.1007 | Accuracy train: 0.9706 | Loss val: 0.1111 | Accuracy val: 0.9682
Época 06 | Loss train: 0.0829 | Accuracy train: 0.9763 | Loss val: 0.0980 | Accuracy val: 0.9712
Época 07 | Loss train: 0.0697 | Accuracy train: 0.9797 | Loss val: 0.0949 | Accuracy val: 0.9715
Época 08 | Loss train: 0.0591 | Accuracy train: 0.9836 | Loss val: 0.0920 | Accuracy val: 0.9721
Época 09 | Loss train: 0.0510 | Accuracy train: 0.9857 | Loss val: 0.0850 | Accuracy val: 0.9750
Época 10 | Loss train: 0.0425 | Accuracy train: 0.9886 | Loss val: 0.0860 | Ac

El mejor modelo se selecciona usando  accuracy .si dos modelos llegaran a empatar en accuracy se usa loss de validación como desempate.

In [42]:
print("\n" + "=" * 80)
print("Resumen de modelos")
print("=" * 80)

for indice, resultado in enumerate(resultados, start=1):
    print(
        f"{indice}. {resultado['nombre']} | "
        f"Mejor época: {resultado['mejorepoca']} | "
        f"Mejor accuracy val: {resultado['mejoraccuracyvalidacion']:.4f} | "
        f"Mejor loss val: {resultado['mejorlossvalidacion']:.4f}"
    )

mejorresultado = resultados[0]

for resultado in resultados[1:]:
    if resultado["mejoraccuracyvalidacion"] > mejorresultado["mejoraccuracyvalidacion"]:
        mejorresultado = resultado
    elif (
        resultado["mejoraccuracyvalidacion"] == mejorresultado["mejoraccuracyvalidacion"]
        and resultado["mejorlossvalidacion"] < mejorresultado["mejorlossvalidacion"]
    ):
        mejorresultado = resultado

print("\nMejor modelo seleccionado:")
print(mejorresultado["nombre"])
print("Configuración:", mejorresultado["configuracion"])


Resumen de modelos
1. Modelo 1 con una capa oculta y Adam | Mejor época: 11 | Mejor accuracy val: 0.9765 | Mejor loss val: 0.0802
2. Modelo 2 con dos capas ocultas y Adam | Mejor época: 234 | Mejor accuracy val: 0.9833 | Mejor loss val: 0.2162
3. Modelo 3 con tres capas ocultas y SGD | Mejor época: 19 | Mejor accuracy val: 0.9839 | Mejor loss val: 0.0937

Mejor modelo seleccionado:
Modelo 3 con tres capas ocultas y SGD
Configuración: {'nombre': 'Modelo 3 con tres capas ocultas y SGD', 'capasocultas': [512, 256, 128], 'optimizador': 'SGD', 'batchsize': 128, 'epocas': 300, 'learningrate': 0.05, 'patience': 4, 'delta': 0.001}


In [43]:
_, _, cargadorprueba = crearcargadores(
    batchsizeentrenamiento=mejorresultado["configuracion"]["batchsize"]
)

funcionperdidafinal = torch.nn.CrossEntropyLoss()

lossprueba, accuracyprueba, prediccionesprueba, etiquetasrealesprueba = evaluarmodelo(
    mejorresultado["modelo"],
    cargadorprueba,
    funcionperdidafinal,
    devolverpredicciones=True
)

print("\n" + "=" * 80)
print("Evaluación final en test")
print("=" * 80)
print(f"Loss test: {lossprueba:.6f}")
print(f"Accuracy test: {accuracyprueba:.6f}")


Evaluación final en test
Loss test: 0.083699
Accuracy test: 0.985100


In [44]:
matrizconfusion = construirmatrizconfusion(
    etiquetasrealesprueba,
    prediccionesprueba,
    numeroclases=10
)

imprimirreporteclasificacion(matrizconfusion)


Matriz de confusión:
tensor([[ 973,    0,    0,    2,    0,    0,    2,    1,    2,    0],
        [   0, 1131,    0,    0,    0,    1,    1,    0,    2,    0],
        [   2,    1, 1017,    2,    2,    0,    1,    3,    4,    0],
        [   1,    0,    2,  991,    0,    4,    0,    4,    7,    1],
        [   1,    0,    1,    0,  967,    0,    5,    2,    0,    6],
        [   2,    0,    0,    8,    0,  875,    3,    0,    3,    1],
        [   2,    2,    0,    1,    2,    3,  947,    0,    1,    0],
        [   0,    3,    7,    2,    0,    0,    0, 1012,    2,    2],
        [   1,    0,    3,    4,    0,    4,    1,    3,  953,    5],
        [   3,    2,    0,    3,    8,    5,    0,    1,    2,  985]])

Accuracy global:
0.985100

Reporte por clase:
Clase     Precision      Recall         F1Score        
0         0.987817       0.992857       0.990331       
1         0.992976       0.996476       0.994723       
2         0.987379       0.985465       0.986421       
3     

In [45]:
torch.save(mejorresultado["modelo"].state_dict(), "mejormodelomnist.pth")
print("Mejor modelo guardado en mejormodelomnist.pth")

Mejor modelo guardado en mejormodelomnist.pth


reflexion personal Hacer las cosas más ordenadas y pensando a futuro, como fue declarar desde antes cómo funcionarían las cosas, me ayudó a que una vez que todo estuvo hecho fuera más sencillo trabajar. Normalmente no suelo ser tan ordenado, pero aquí, para evitar futuras confusiones, tomé la decisión desde el inicio de usar palabras en español para saber mejor qué estoy haciendo bien o mal. Esto fue por la limitacion de pytorch Las partes en inglés las dejé así porque usar palabras como size, loss y accuracy en español sí me resultaría muy extraño pero en general la distincion de idioma fue para no perder el flujo pues estoy haciendo esta actividad en la madrugada y esto me sirviria mas para recordar que funcion tenia cada funcion y variable rapidamente. Creí que lo más complicado sería la matriz, pero recordé que en esencia sigue siendo una matriz, así que no era necesario que fuera de colores o con formas llamativas.
Decidí usar una cantidad exagerada de épocas para forzar el detenimiento temprano, porque originalmente lo había hecho con una cantidad entre 100 y solo uno de los modelos se detuvo. En el caso del modelo de tres capas ocultas, eso se notó más. Aprendí que una red neuronal no es tan complicada como yo creía. Por ejemplo, antes veía videos de inteligencia artificial que aprende a jugar cierto juego y lo imaginaba como algo mucho más complejo creo que implementarlo en un juego seria complejo por como lo usarias y que equivaldria a que pero en si la red no seria algo que no pudiera comprender aun que este ejercicio entiendo que es para entender las bases la misma clase vimos una extencion tanto de complejidad y logica. Esta actividad me ayudó comprender las bases que eran más simples de lo que pensaba, o al menos me ayudó a perderles el miedo. Algo que me sorprendió bastante fue darme cuenta de que más épocas no significa realmente un cambio o una mejora tan significativa, al menos no en estos casos. El resultado final siguió favoreciendo al tercer modelo, que además se detuvo de manera temprana. En cambio, el segundo hizo todas las épocas y no se detuvo ni pasó algo similar, y aun así no fue el mejor. Eso me dejó muy impactado. Creo que para la próxima clase tengo algunas dudas, porque todo este tiempo pensé que un mismo modelo, si evitaba el sobreentrenamiento y otros problemas, con muchísima potencia de cómputo y mucho tiempo siempre sería mejor que ese mismo modelo entrenado con una potencia más decente y en un tiempo razonable.